In [10]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy import stats
from scipy.signal import welch
import os
import glob

In [11]:
class RGBMentalPerformancePredictor:

    def __init__(self, r_path, g_path, b_path):
        self.r_path = r_path
        self.g_path = g_path
        self.b_path = b_path
        self.channel_paths = {'R': r_path, 'G': g_path, 'B': b_path}
        self.scaler = StandardScaler()
        self.model = None

    def load_data(self):
        print("Loading data from RGB channels...")

        data_list = []

        chunk_files = glob.glob(os.path.join(self.r_path, "*chunks*.csv"))

        participant_ids = set()
        for file in chunk_files:
            filename = os.path.basename(file)
            import re
            match = re.search(r'chunks[_]?(\d+)', filename)
            if match:
                participant_ids.add(match.group(1))

        participant_ids = sorted(list(participant_ids))
        print(f"Found {len(participant_ids)} participants: {participant_ids}")

        for participant_id in participant_ids:
            print(f"Processing participant {participant_id}...")

            channel_data = {}

            for channel_name, channel_path in self.channel_paths.items():
                pattern = os.path.join(channel_path, f"*chunks*{participant_id}.csv")
                chunk_files = sorted(glob.glob(pattern))

                if not chunk_files:
                    print(f"  Warning: No chunks found for participant {participant_id} in channel {channel_name}")
                    continue

                print(f"  Channel {channel_name}: {len(chunk_files)} chunk(s)")

                chunk_dfs = []
                for chunk_file in chunk_files:
                    df_chunk = pd.read_csv(chunk_file)
                    chunk_dfs.append(df_chunk)

                channel_data[channel_name] = pd.concat(chunk_dfs, ignore_index=True)
                print(f"    Total rows: {len(channel_data[channel_name])}")

            if len(channel_data) == 3:
                lengths = [len(channel_data[ch]) for ch in ['R', 'G', 'B']]
                if len(set(lengths)) != 1:
                    print(f"  Warning: Channels have different lengths: {lengths}")
                    min_length = min(lengths)
                    print(f"  Trimming to minimum length: {min_length}")
                    for ch in ['R', 'G', 'B']:
                        channel_data[ch] = channel_data[ch].iloc[:min_length]

            num_rows = len(channel_data['R'])
            for idx in range(num_rows):
                features = {}

                for channel_name in ['R', 'G', 'B']:
                    time_series_cols = [f't{i}' for i in range(60)]
                    time_series = channel_data[channel_name].iloc[idx][time_series_cols].values.astype(float)

                    channel_features = self.extract_features(time_series, channel_name)
                    features.update(channel_features)

                try:
                    target = channel_data['R'].iloc[idx]['AU']
                    if pd.isna(target):
                        continue
                except (KeyError, IndexError):
                    print(f"  Warning: Could not get AU value for row {idx}")
                    continue

                features['participant_id'] = participant_id
                features['target'] = target
                data_list.append(features)

        self.df = pd.DataFrame(data_list)
        print(f"\n✓ Successfully loaded {len(self.df)} total samples from {len(participant_ids)} participants")
        return self.df

    def extract_features(self, time_series, channel_name):
        """Extract statistical, temporal, and frequency features from time series."""
        features = {}
        prefix = f"{channel_name}_"

        features[f'{prefix}mean'] = np.mean(time_series)
        features[f'{prefix}std'] = np.std(time_series)
        features[f'{prefix}min'] = np.min(time_series)
        features[f'{prefix}max'] = np.max(time_series)
        features[f'{prefix}median'] = np.median(time_series)
        features[f'{prefix}range'] = np.ptp(time_series)
        features[f'{prefix}skewness'] = stats.skew(time_series)
        features[f'{prefix}kurtosis'] = stats.kurtosis(time_series)

        features[f'{prefix}p25'] = np.percentile(time_series, 25)
        features[f'{prefix}p75'] = np.percentile(time_series, 75)
        features[f'{prefix}iqr'] = features[f'{prefix}p75'] - features[f'{prefix}p25']

        diff = np.diff(time_series)
        features[f'{prefix}mean_diff'] = np.mean(diff)
        features[f'{prefix}std_diff'] = np.std(diff)
        features[f'{prefix}mean_abs_diff'] = np.mean(np.abs(diff))

        features[f'{prefix}zero_crossing'] = np.sum(np.diff(np.sign(time_series - np.mean(time_series))) != 0)

        x = np.arange(len(time_series))
        slope, intercept, _, _, _ = stats.linregress(x, time_series)
        features[f'{prefix}trend_slope'] = slope
        features[f'{prefix}trend_intercept'] = intercept

        freqs, psd = welch(time_series, fs=1.0, nperseg=min(len(time_series), 30))
        features[f'{prefix}psd_mean'] = np.mean(psd)
        features[f'{prefix}psd_std'] = np.std(psd)
        features[f'{prefix}psd_max'] = np.max(psd)
        features[f'{prefix}dominant_freq'] = freqs[np.argmax(psd)]

        features[f'{prefix}energy'] = np.sum(time_series ** 2)

        return features

    def prepare_data(self):
        print("\nPreparing data...")

        print(f"Total samples before cleaning: {len(self.df)}")
        print(f"Missing values in target: {self.df['target'].isna().sum()}")

        self.df = self.df.dropna(subset=['target'])
        print(f"Total samples after removing missing targets: {len(self.df)}")

        feature_cols = [col for col in self.df.columns
                       if col not in ['target', 'participant_id']]

        X = self.df[feature_cols]
        y = self.df['target']

        missing_features = X.isna().sum()
        if missing_features.sum() > 0:
            print(f"\nWarning: Found {missing_features.sum()} missing values in features")
            print("Filling missing values with column means...")
            X = X.fillna(X.mean())

        inf_mask = np.isinf(X.values)
        if inf_mask.any():
            print(f"Warning: Found {inf_mask.sum()} infinite values in features")
            print("Replacing infinite values with column max/min...")
            X = X.replace([np.inf, -np.inf], np.nan)
            X = X.fillna(X.mean())

        print(f"\n✓ Number of features: {len(feature_cols)}")
        print(f"✓ Target range: [{y.min():.2f}, {y.max():.2f}]")
        print(f"✓ Target mean: {y.mean():.2f}, std: {y.std():.2f}")

        return X, y

    def train_and_evaluate_models(self, X, y, test_size=0.2, random_state=42):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state
        )

        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)

        models = {
            'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10,
                                                   random_state=random_state, n_jobs=-1),
            'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5,
                                                           learning_rate=0.1, random_state=random_state),
            'Ridge Regression': Ridge(alpha=1.0),
            'Lasso Regression': Lasso(alpha=0.1, max_iter=5000),
            'SVR (RBF)': SVR(kernel='rbf', C=10, gamma='scale')
        }

        results = {}

        print("MENTAL PERFORMANCE PREDICTION RESULTS")
        print("Predicting mental performance (AU) from RGB video signals")

        info_line = f"║  Train samples: {len(X_train):>5} | Test samples: {len(X_test):>5} | Features: {X.shape[1]:>3}"
        info_line = info_line + " "*(90 - len(info_line))
        print(info_line)

        for name, model in models.items():

            print(f"MODEL: {name:<79}")

            model.fit(X_train_scaled, y_train)

            y_train_pred = model.predict(X_train_scaled)
            y_test_pred = model.predict(X_test_scaled)

            train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
            test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
            train_mae = mean_absolute_error(y_train, y_train_pred)
            test_mae = mean_absolute_error(y_test, y_test_pred)
            train_r2 = r2_score(y_train, y_train_pred)
            test_r2 = r2_score(y_test, y_test_pred)

            cv_scores = cross_val_score(model, X_train_scaled, y_train,
                                       cv=5, scoring='neg_root_mean_squared_error')
            cv_rmse = -cv_scores.mean()
            cv_std = cv_scores.std()

            results[name] = {
                'model': model,
                'train_rmse': train_rmse,
                'test_rmse': test_rmse,
                'train_mae': train_mae,
                'test_mae': test_mae,
                'train_r2': train_r2,
                'test_r2': test_r2,
                'cv_rmse': cv_rmse,
                'cv_std': cv_std
            }


            mae_percentage = (test_mae / y.mean()) * 100

            print(f" ACCURACY METRICS")
            print(f"   Test RMSE:  {test_rmse:>7.3f}  (average error in AU units)")
            print(f"   Test MAE:   {test_mae:>7.3f}  ({mae_percentage:.1f}% of mean AU)")
            print(f" Test R²:    {test_r2:>7.3f}  ({test_r2*100:.1f}% variance explained)")

            print(f" MODEL RELIABILITY")
            print(f"CV RMSE:    {cv_rmse:>7.3f} ± {cv_std:.3f}  (5-fold cross-validation)")

            overfitting = train_r2 - test_r2
            if overfitting > 0.15:
                status = "High overfitting detected"
            elif overfitting > 0.05:
                status = "Slight overfitting"
            else:
                status = "Good generalization"

            print(f"Train R²:   {train_r2:>7.3f}  ({status})")


            if test_r2 > 0.8:
                rating = "Excellent"
                interp = "Strong predictive power"
            elif test_r2 > 0.6:
                rating = "Good"
                interp = "Solid performance"
            elif test_r2 > 0.4:
                rating = "Moderate"
                interp = "Acceptable but improvable"
            else:
                rating = "Weak"
                interp = "Needs improvement"

            print(f"INTERPRETATION")
            print(f"Rating: {rating:<15} {interp:<40}")
            print(f"Prediction: Average error of {test_mae:.2f} AU units in mental performance")


        best_model_name = min(results.items(), key=lambda x: x[1]['test_rmse'])[0]
        self.model = results[best_model_name]['model']
        best_results = results[best_model_name]

        print("BEST MODEL")

        print(f"  Model:              {best_model_name:<65}  ")
        print(f"  Test RMSE:          {best_results['test_rmse']:.4f} AU units{' '*51}")
        print(f"  Test MAE:           {best_results['test_mae']:.4f} AU units{' '*51}")
        print(f"  Test R²:            {best_results['test_r2']:.4f} ({best_results['test_r2']*100:.1f}% variance explained){' '*35}")


        print(" CLINICAL INTERPRETATION")


        if best_results['test_mae'] < 5:
            precision = "high precision"
        elif best_results['test_mae'] < 10:
            precision = "moderate precision"
        else:
            precision = "lower precision"

        print(f"  This model predicts mental performance with {precision}. On average,")
        print(f"  predictions are off by {best_results['test_mae']:.2f} AU units. The model captures{' '*20}")
        print(f"  {best_results['test_r2']*100:.1f}% of the variance in mental performance from RGB signals.{' '*23}")

        if best_results['test_r2'] > 0.7:
            print("  Conclusion: Suitable for research and potential clinical screening tools.")
        elif best_results['test_r2'] > 0.5:
            print("  Conclusion: Shows promise but may need refinement for clinical use.")
        else:
            print("  Conclusion: Consider additional features or different modeling approaches.")


        return results, X_test_scaled, y_test

In [12]:
if __name__ == "__main__":
    r_path = "/content/R"
    g_path = "/content/G"
    b_path = "/content/B"

    predictor = RGBMentalPerformancePredictor(r_path, g_path, b_path)

    df = predictor.load_data()
    X, y = predictor.prepare_data()

    results, X_test, y_test = predictor.train_and_evaluate_models(X, y)

    best_model_name = min(results.items(), key=lambda x: x[1]['test_rmse'])[0]
    best_model = results[best_model_name]['model']

    print(f"✓ Best model: {best_model_name}")

Loading data from RGB channels...
Found 10 participants: ['10', '11', '12', '3', '4', '5', '6', '7', '8', '9']
Processing participant 10...
  Channel R: 1 chunk(s)
    Total rows: 135
  Channel G: 1 chunk(s)
    Total rows: 135
  Channel B: 1 chunk(s)
    Total rows: 135
Processing participant 11...
  Channel R: 1 chunk(s)
    Total rows: 87
  Channel G: 1 chunk(s)
    Total rows: 87
  Channel B: 1 chunk(s)
    Total rows: 87
Processing participant 12...
  Channel R: 1 chunk(s)
    Total rows: 85
  Channel G: 1 chunk(s)
    Total rows: 85
  Channel B: 1 chunk(s)
    Total rows: 85
Processing participant 3...
  Channel R: 1 chunk(s)
    Total rows: 92
  Channel G: 1 chunk(s)
    Total rows: 92
  Channel B: 1 chunk(s)
    Total rows: 92
Processing participant 4...
  Channel R: 1 chunk(s)
    Total rows: 74
  Channel G: 1 chunk(s)
    Total rows: 74
  Channel B: 1 chunk(s)
    Total rows: 74
Processing participant 5...
  Channel R: 1 chunk(s)
    Total rows: 120
  Channel G: 1 chunk(s)
  